# Image Pre-processing for Alzheimer's Disease MRI Classification

This notebook handles dataset collection and pre-processing for the OTSU-GWWHO framework. It loads the OASIS MRI slices from the four class folders (MD, MOD, ND, VMD), validates each image, converts it to grayscale, resizes it to 224×224, and applies min-max intensity normalisation to the 8-bit range [0, 255] so that Otsu thresholding works on L = 256 grey levels. The dataset is then divided with a stratified 80:20 training–testing split.

## Prerequisites

- Python 3.10+
- Required libraries: `numpy`, `opencv-python`, `pandas`, `scikit-learn`, `tqdm`
- Dataset from https://www.kaggle.com/datasets/ninadaithal/imagesoasis organised in `./dataset/Data/` with one sub-folder per class

## Workflow

1. Map the dataset folders to the class labels MD, MOD, ND and VMD
2. Validate each image (skip unreadable, empty or constant images)
3. Convert to grayscale and resize to 224×224
4. Normalise intensities to [0, 255]
5. Save pre-processed images and a manifest file
6. Perform a stratified 80:20 training–testing split

In [ ]:
# ==========================================================
# IMPORTS & CONFIGURATION
# ==========================================================

import os
import re
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

RAW_DATA_DIR = "./dataset/Data"
PREPROCESSED_DIR = "./outputs/01_preprocessed"
SPLIT_DIR = "./outputs/02_splits"

CLASS_FOLDER_MAP = {
    "Mild Dementia": "MD",
    "Moderate Dementia": "MOD",
    "Non Demented": "ND",
    "Very mild Dementia": "VMD",
}
CLASS_NAMES = ["MD", "MOD", "ND", "VMD"]

IMG_SIZE = 224          # VGG16 input size
GRAY_LEVELS = 256       # L intensity levels for Otsu
TEST_SIZE = 0.20        # 80% training / 20% testing
SPLIT_BY_SUBJECT = False  # True -> all slices of one subject stay in the same split
RANDOM_SEED = 42
VALID_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

## Image Validation and Subject ID

Utility functions to reject invalid images and to read the OASIS subject ID from the file name (e.g. `OAS1_0001_MR1_mpr-1_100.jpg` → `OAS1_0001`).

In [ ]:
# ==========================================================
# IMAGE VALIDATION
# ==========================================================

def validate_image(img):
    """Reject unreadable, empty or constant images."""
    if img is None or img.size == 0 or img.ndim != 2:
        return False
    return float(img.max()) > float(img.min())


def extract_subject_id(filename):
    match = re.match(r"(OAS\d_\d{4})", filename)
    return match.group(1) if match else os.path.splitext(filename)[0]

## Resizing and Intensity Normalisation

Each MRI slice is resized to 224×224 and min-max normalised to the full 8-bit range [0, L−1].

In [ ]:
# ==========================================================
# PRE-PROCESSING FUNCTION
# ==========================================================

def normalize_intensity(img):
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return np.round(img * (GRAY_LEVELS - 1)).astype(np.uint8)


def preprocess_image(path):
    """Load -> validate -> grayscale -> resize -> normalise."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if not validate_image(img):
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return normalize_intensity(img)

## Process the Complete Dataset

Pre-processes every image of the four classes and writes `manifest.csv` (image path, label and subject ID).

In [ ]:
# ==========================================================
# PROCESS DATASET
# ==========================================================

records, skipped = [], 0

for folder, label in CLASS_FOLDER_MAP.items():
    class_in = os.path.join(RAW_DATA_DIR, folder)
    class_out = os.path.join(PREPROCESSED_DIR, label)
    os.makedirs(class_out, exist_ok=True)

    files = sorted(f for f in os.listdir(class_in) if f.lower().endswith(VALID_EXT))
    for fname in tqdm(files, desc=f"Pre-processing {label}"):
        img = preprocess_image(os.path.join(class_in, fname))
        if img is None:
            skipped += 1
            continue
        stem = os.path.splitext(fname)[0]
        out_path = os.path.join(class_out, stem + ".png")
        cv2.imwrite(out_path, img)
        records.append({"image_id": f"{label}/{stem}", "path": out_path,
                        "label": label, "subject_id": extract_subject_id(fname)})

manifest = pd.DataFrame(records)
manifest.to_csv(os.path.join(PREPROCESSED_DIR, "manifest.csv"), index=False)

counts = manifest["label"].value_counts().reindex(CLASS_NAMES)
print(f"Saved {len(manifest)} images, skipped {skipped} invalid images")
print(pd.DataFrame({"No. of Images": counts,
                    "Image Proportion (%)": (100 * counts / counts.sum()).round(2)}))

## Training–Testing Split

Stratified 80:20 split of the original images. The 20% test set is never augmented. Set `SPLIT_BY_SUBJECT = True` for a subject-level split.

In [ ]:
# ==========================================================
# 80:20 STRATIFIED SPLIT
# ==========================================================

if SPLIT_BY_SUBJECT:
    sgkf = StratifiedGroupKFold(n_splits=int(1 / TEST_SIZE), shuffle=True, random_state=RANDOM_SEED)
    tr_idx, te_idx = next(sgkf.split(manifest, manifest["label"], manifest["subject_id"]))
    train_df, test_df = manifest.iloc[tr_idx], manifest.iloc[te_idx]
else:
    train_df, test_df = train_test_split(manifest, test_size=TEST_SIZE,
                                         stratify=manifest["label"], random_state=RANDOM_SEED)

os.makedirs(SPLIT_DIR, exist_ok=True)
train_df.reset_index(drop=True).to_csv(os.path.join(SPLIT_DIR, "train.csv"), index=False)
test_df.reset_index(drop=True).to_csv(os.path.join(SPLIT_DIR, "test.csv"), index=False)

table = pd.DataFrame({"Training (80%)": train_df["label"].value_counts(),
                      "Testing (20%)": test_df["label"].value_counts()}).reindex(CLASS_NAMES)
table["Total"] = table.sum(axis=1)
print(table)